In [11]:
# Enhanced Fingerprint Classification with Advanced Ensemble - Final Version
# Target: >90% accuracy with comprehensive analysis
# Preserves all core logic and fixes from the original code

# ============================================
# SECTION 1: INITIAL SETUP AND IMPORTS
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

# TensorFlow and Keras imports
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization, Layer
from tensorflow.keras.layers import Lambda, Add, Multiply
from tensorflow.keras.applications import EfficientNetB0, InceptionV3
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, Callback
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers

# For evaluation and analysis
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import label_binarize
import cv2
from scipy import stats

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# ============================================
# SECTION 2: MOUNT GOOGLE DRIVE
# ============================================

from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/NISTDB4_RAW'

# Define paths
TRAIN_PATH = os.path.join(BASE_PATH, 'train_set')
VAL_PATH = os.path.join(BASE_PATH, 'val_set')
TEST_PATH = os.path.join(BASE_PATH, 'test_set')

# Verify paths and count samples
print("Checking dataset paths...")
dataset_info = {}
for path_name, path in [("Train", TRAIN_PATH), ("Validation", VAL_PATH), ("Test", TEST_PATH)]:
    if os.path.exists(path):
        print(f"✓ {path_name} path exists: {path}")
        class_counts = {}
        for class_dir in os.listdir(path):
            class_path = os.path.join(path, class_dir)
            if os.path.isdir(class_path):
                num_images = len([f for f in os.listdir(class_path) if f.endswith(('.png', '.jpg', '.jpeg'))])
                class_counts[class_dir] = num_images
                print(f"  - {class_dir}: {num_images} images")
        dataset_info[path_name] = class_counts
    else:
        print(f"✗ {path_name} path NOT found: {path}")


TensorFlow version: 2.19.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checking dataset paths...
✓ Train path exists: /content/drive/MyDrive/NISTDB4_RAW/train_set
  - class3_loop: 560 images
  - class2_whorl: 559 images
  - class1_arc: 560 images
✓ Validation path exists: /content/drive/MyDrive/NISTDB4_RAW/val_set
  - class1_arc: 160 images
  - class3_loop: 160 images
  - class2_whorl: 159 images
✓ Test path exists: /content/drive/MyDrive/NISTDB4_RAW/test_set
  - class2_whorl: 81 images
  - class1_arc: 81 images
  - class3_loop: 81 images


In [12]:
# ============================================
# SECTION 3: ENHANCED CONFIGURATION
# ============================================

IMG_SIZE = 224
BATCH_SIZE = 32
INITIAL_EPOCHS = 30  # Increased for better convergence
FINE_TUNE_EPOCHS = 25  # More fine-tuning
NUM_CLASSES = 3
ENSEMBLE_WEIGHTS = {'efficientnet': 0.55, 'inception': 0.45}  # Optimized weights

# Create directories
os.makedirs('/content/models', exist_ok=True)
os.makedirs('/content/results', exist_ok=True)
os.makedirs('/content/analysis', exist_ok=True)

# ============================================
# SECTION 4: ADVANCED AUGMENTATION FUNCTIONS
# ============================================

def random_cutout(image, min_holes=1, max_holes=5, min_size=0.1, max_size=0.25):
    """Enhanced cutout with adaptive sizing based on image content"""
    h, w, _ = image.shape

    # Calculate image complexity (edge density)
    gray = cv2.cvtColor(image.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    edge_density = np.sum(edges) / (h * w * 255)

    # Adjust cutout based on complexity
    if edge_density > 0.1:  # High detail image
        max_size *= 0.7  # Smaller cutouts for detailed images

    for _ in range(np.random.randint(min_holes, max_holes + 1)):
        hole_h = int(h * np.random.uniform(min_size, max_size))
        hole_w = int(w * np.random.uniform(min_size, max_size))
        y1 = np.random.randint(0, h - hole_h)
        x1 = np.random.randint(0, w - hole_w)
        y2 = y1 + hole_h
        x2 = x1 + hole_w

        # Use mean pixel value instead of black for less disruption
        mean_val = np.mean(image[y1:y2, x1:x2, :], axis=(0, 1))
        image[y1:y2, x1:x2, :] = mean_val

    return image

def random_gaussian_blur(image, max_kernel_size=5):
    """Adaptive Gaussian blur based on image sharpness"""
    # Calculate image sharpness using Laplacian
    gray = cv2.cvtColor(image.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()

    # Adjust blur based on sharpness
    if laplacian_var > 100:  # Sharp image
        kernel_size = np.random.randint(1, min(3, max_kernel_size) + 1)
    else:  # Already blurry
        kernel_size = 1  # Minimal blur

    if kernel_size % 2 == 0:
        kernel_size += 1

    if kernel_size > 1:
        return cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)
    return image

def mixup_augmentation(image1, image2, alpha=0.2):
    """MixUp augmentation for better generalization"""
    lam = np.random.beta(alpha, alpha)
    return lam * image1 + (1 - lam) * image2

def augment_image(image):
    """Enhanced augmentation pipeline with probability-based application"""
    # Apply Gaussian Blur adaptively
    if np.random.rand() < 0.25:
        image = random_gaussian_blur(image)

    # Apply Cutout with adjusted probability
    if np.random.rand() < 0.35:
        image = random_cutout(image)

    # Add slight noise for robustness
    if np.random.rand() < 0.15:
        noise = np.random.normal(0, 5, image.shape)
        image = np.clip(image + noise, 0, 255)

    return image

In [13]:
# ============================================
# SECTION 5: TEST-TIME AUGMENTATION (TTA)
# ============================================

class TestTimeAugmentation:
    """Implement TTA for better predictions"""

    def __init__(self, model, augmentations=5):
        self.model = model
        self.augmentations = augmentations

    def predict(self, image):
        """Predict with multiple augmentations and average"""
        predictions = []

        # Original prediction
        predictions.append(self.model.predict(np.expand_dims(image, axis=0), verbose=0))

        # Augmented predictions
        for _ in range(self.augmentations - 1):
            aug_image = image.copy()

            # Random transformations
            if np.random.rand() > 0.5:
                aug_image = np.fliplr(aug_image)

            angle = np.random.uniform(-10, 10)
            M = cv2.getRotationMatrix2D((IMG_SIZE//2, IMG_SIZE//2), angle, 1)
            aug_image = cv2.warpAffine(aug_image, M, (IMG_SIZE, IMG_SIZE))

            predictions.append(self.model.predict(np.expand_dims(aug_image, axis=0), verbose=0))

        # Average predictions
        return np.mean(predictions, axis=0)

In [14]:
# ============================================
# SECTION 6: ENHANCED DATA GENERATORS
# ============================================

# Training generator with advanced augmentation
train_datagen = ImageDataGenerator(
    rotation_range=25,  # Increased rotation
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest',
    preprocessing_function=augment_image
)

# Validation and test generators
val_test_datagen = ImageDataGenerator()

print("\nCreating data generators...")

train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

validation_generator = val_test_datagen.flow_from_directory(
    VAL_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Identify class indices (preserving original logic)
loop_class_index = train_generator.class_indices.get('class3_loop', -1)
arc_class_index = train_generator.class_indices.get('class1_arc', -1)

print(f"\nClass indices identified:")
print(f"Loop class index: {loop_class_index}")
print(f"Arc class index: {arc_class_index}")


Creating data generators...
Found 1679 images belonging to 3 classes.
Found 479 images belonging to 3 classes.
Found 243 images belonging to 3 classes.

Class indices identified:
Loop class index: 2
Arc class index: 0


In [15]:
# ============================================
# SECTION 7: OPTIMIZED CLASS WEIGHTS
# ============================================

labels = train_generator.labels
class_weights_array = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weights = {i: w for i, w in enumerate(class_weights_array)}

# Enhanced weighting strategy
if loop_class_index != -1:
    class_weights[loop_class_index] *= 1.4  # Increased from 1.75
    print("✓ Loop class priority weight applied")

if arc_class_index != -1:
    class_weights[arc_class_index] *= 1.15  # Add arc weight boost
    print("✓ Arc class weight boost applied")

print("\nOptimized Class Weights:")
for idx, weight in class_weights.items():
    class_name = [k for k,v in train_generator.class_indices.items() if v == idx][0]
    print(f"  {class_name}: {weight:.3f}")


✓ Loop class priority weight applied
✓ Arc class weight boost applied

Optimized Class Weights:
  class1_arc: 1.149
  class2_whorl: 1.001
  class3_loop: 1.399


In [16]:
# ============================================
# SECTION 8: ADVANCED MODEL ARCHITECTURE
# ============================================

# Preserve the original preprocessing wrapper functions
def eff_preprocess_wrapper(x):
    """Wrapper for the EfficientNet preprocessing function."""
    return efficientnet_preprocess(x)

def inc_preprocess_wrapper(x):
    """Wrapper for the InceptionV3 preprocessing function."""
    return inception_preprocess(x)

def build_model(model_name="efficientnet"):
    """Enhanced model architecture with attention mechanism"""
    inputs = tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    if model_name == "efficientnet":
        x = Lambda(eff_preprocess_wrapper, name='efficientnet_preprocessing')(inputs)
        base_model = EfficientNetB0(input_tensor=x, include_top=False, weights='imagenet')
    elif model_name == "inceptionv3":
        x = Lambda(inc_preprocess_wrapper, name='inceptionv3_preprocessing')(inputs)
        base_model = InceptionV3(input_tensor=x, include_top=False, weights='imagenet')
    else:
        raise ValueError("model_name must be 'efficientnet' or 'inceptionv3'")

    base_model.trainable = False

    # Enhanced classification head with attention
    x = base_model.output

    # Global Average Pooling
    gap = GlobalAveragePooling2D()(x)

    # Attention mechanism
    attention = Dense(1280 if model_name == "efficientnet" else 2048, activation='sigmoid')(gap)
    x = Multiply()([gap, attention])

    # Classification layers
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)  # Increased dropout
    x = Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model, base_model

In [17]:
# ============================================
# SECTION 9: ADVANCED TRAINING WITH MONITORING
# ============================================

class PerformanceMonitor(Callback):
    """Custom callback to monitor per-class performance"""
    def __init__(self, validation_data, class_names):
        super().__init__()
        self.validation_data = validation_data
        self.class_names = class_names
        self.history = {'epoch': [], 'class_accuracies': {}}
        for class_name in class_names:
            self.history['class_accuracies'][class_name] = []

    def on_epoch_end(self, epoch, logs=None):
        # Get predictions
        predictions = self.model.predict(self.validation_data, verbose=0)
        predicted_classes = np.argmax(predictions, axis=1)
        true_classes = self.validation_data.classes

        # Calculate per-class accuracy
        self.history['epoch'].append(epoch)
        for i, class_name in enumerate(self.class_names):
            mask = true_classes == i
            if np.sum(mask) > 0:
                class_acc = np.mean(predicted_classes[mask] == i)
                self.history['class_accuracies'][class_name].append(class_acc)
                print(f"  {class_name} accuracy: {class_acc:.4f}")

def train_model_advanced(model, base_model, model_name, train_gen, val_gen, c_weights, initial_epochs, fine_tune_epochs):
    """Enhanced training function with advanced monitoring"""

    # Get class names
    class_names = list(train_gen.class_indices.keys())

    # Performance monitor
    perf_monitor = PerformanceMonitor(val_gen, class_names)

    # Stage 1: Train Head
    print(f"\n{'='*50}\nTraining {model_name} - Stage 1: Classification Head\n{'='*50}")

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )

    callbacks_s1 = [
        ModelCheckpoint(f'/content/models/best_model_{model_name}_s1.h5',
                       monitor='val_accuracy', save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-7, verbose=1),
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
        perf_monitor
    ]

    history_s1 = model.fit(
        train_gen, epochs=initial_epochs, validation_data=val_gen,
        callbacks=callbacks_s1, class_weight=c_weights, verbose=1
    )

    # Stage 2: Fine-Tuning
    print(f"\n{'='*50}\nTraining {model_name} - Stage 2: Fine-Tuning\n{'='*50}")

    base_model.trainable = True
    fine_tune_at = int(len(base_model.layers) * 0.65)  # Unfreeze more layers
    for layer in base_model.layers[:fine_tune_at]:
        layer.trainable = False

    model.compile(
        optimizer=Adam(learning_rate=5e-6),  # Lower learning rate
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )

    callbacks_s2 = [
        ModelCheckpoint(f'/content/models/best_model_{model_name}_s2_finetuned.h5',
                       monitor='val_accuracy', save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-8, verbose=1),
        EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
        perf_monitor
    ]

    history_s2 = model.fit(
        train_gen, epochs=fine_tune_epochs, validation_data=val_gen,
        callbacks=callbacks_s2, class_weight=c_weights, verbose=1
    )

    return history_s1, history_s2, perf_monitor.history

In [18]:
# ============================================
# SECTION 10: TRAIN BOTH MODELS
# ============================================

# Train EfficientNet
efficientnet_model, efficientnet_base = build_model("efficientnet")
hist_eff_s1, hist_eff_s2, eff_class_history = train_model_advanced(
    efficientnet_model, efficientnet_base, "efficientnet",
    train_generator, validation_generator, class_weights,
    INITIAL_EPOCHS, FINE_TUNE_EPOCHS
)

# Train InceptionV3
inception_model, inception_base = build_model("inceptionv3")
hist_inc_s1, hist_inc_s2, inc_class_history = train_model_advanced(
    inception_model, inception_base, "inceptionv3",
    train_generator, validation_generator, class_weights,
    INITIAL_EPOCHS, FINE_TUNE_EPOCHS
)


Training efficientnet - Stage 1: Classification Head
Epoch 1/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 783ms/step - accuracy: 0.5056 - loss: 12.0607 - precision_4: 0.5244 - recall_4: 0.4734
Epoch 1: val_accuracy improved from -inf to 0.46555, saving model to /content/models/best_model_efficientnet_s1.h5


  class1_arc accuracy: 0.0312
  class2_whorl accuracy: 0.3774
  class3_loop accuracy: 0.9875
53/53 ━━━━━━━━━━━━━━━━━━━━ 106s 2s/step - accuracy: 0.5068 - loss: 12.0511 - precision_4: 0.5257 - recall_4: 0.4747 - val_accuracy: 0.4656 - val_loss: 10.4649 - val_precision_4: 0.5145 - val_recall_4: 0.3716 - learning_rate: 0.0010
Epoch 2/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 564ms/step - accuracy: 0.6854 - loss: 10.2877 - precision_4: 0.6952 - recall_4: 0.6643
Epoch 2: val_accuracy did not improve from 0.46555
  class1_arc accuracy: 0.0000
  class2_whorl accuracy: 0.2453
  class3_loop accuracy: 1.0000
53/53 ━━━━━━━━━━━━━━━━━━━━ 82s 649ms/step - accuracy: 0.6851 - loss: 10.2823 - precision_4: 0.6949 - recall_4: 0.6640 - val_accuracy: 0.4154 - val_loss: 9.2639 - val_precision_4: 0.4077 - val_recall_4: 0.3967 - learning_rate: 0.0010
Epoch 3/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 562ms/step - accuracy: 0.6805 - loss: 8.9466 - precision_4: 0.6971 - recall_4: 0.6677
Epoch 3: val_accuracy did not improve from 

  class1_arc accuracy: 0.4125
  class2_whorl accuracy: 0.3962
  class3_loop accuracy: 0.9875
53/53 ━━━━━━━━━━━━━━━━━━━━ 35s 655ms/step - accuracy: 0.7157 - loss: 7.6475 - precision_4: 0.7263 - recall_4: 0.7022 - val_accuracy: 0.5992 - val_loss: 6.7606 - val_precision_4: 0.5969 - val_recall_4: 0.5658 - learning_rate: 0.0010
Epoch 5/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 582ms/step - accuracy: 0.7558 - loss: 6.5647 - precision_4: 0.7663 - recall_4: 0.7492
Epoch 5: val_accuracy did not improve from 0.59916
  class1_arc accuracy: 0.1812
  class2_whorl accuracy: 0.5157
  class3_loop accuracy: 0.9938
53/53 ━━━━━━━━━━━━━━━━━━━━ 35s 666ms/step - accuracy: 0.7555 - loss: 6.5600 - precision_4: 0.7660 - recall_4: 0.7488 - val_accuracy: 0.5637 - val_loss: 5.8359 - val_precision_4: 0.5608 - val_recall_4: 0.5491 - learning_rate: 0.0010
Epoch 6/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 554ms/step - accuracy: 0.7516 - loss: 5.6068 - precision_4: 0.7615 - recall_4: 0.7380
Epoch 6: val_accuracy improved from 0.59916 t

  class1_arc accuracy: 0.5750
  class2_whorl accuracy: 0.7170
  class3_loop accuracy: 0.9688
53/53 ━━━━━━━━━━━━━━━━━━━━ 35s 665ms/step - accuracy: 0.7514 - loss: 5.6034 - precision_4: 0.7613 - recall_4: 0.7378 - val_accuracy: 0.7537 - val_loss: 4.8024 - val_precision_4: 0.7575 - val_recall_4: 0.7370 - learning_rate: 0.0010
Epoch 7/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 559ms/step - accuracy: 0.7370 - loss: 4.8352 - precision_4: 0.7459 - recall_4: 0.7198
Epoch 7: val_accuracy did not improve from 0.75365
  class1_arc accuracy: 0.5813
  class2_whorl accuracy: 0.6730
  class3_loop accuracy: 0.9625
53/53 ━━━━━━━━━━━━━━━━━━━━ 34s 642ms/step - accuracy: 0.7371 - loss: 4.8314 - precision_4: 0.7461 - recall_4: 0.7200 - val_accuracy: 0.7390 - val_loss: 4.1005 - val_precision_4: 0.7516 - val_recall_4: 0.7203 - learning_rate: 0.0010
Epoch 8/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 579ms/step - accuracy: 0.7699 - loss: 4.0409 - precision_4: 0.7802 - recall_4: 0.7628
Epoch 8: val_accuracy did not improve from 0.

  class1_arc accuracy: 0.7500
  class2_whorl accuracy: 0.6415
  class3_loop accuracy: 0.9625
53/53 ━━━━━━━━━━━━━━━━━━━━ 42s 670ms/step - accuracy: 0.7726 - loss: 2.9511 - precision_4: 0.7803 - recall_4: 0.7543 - val_accuracy: 0.7850 - val_loss: 2.5135 - val_precision_4: 0.7886 - val_recall_4: 0.7787 - learning_rate: 0.0010
Epoch 11/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 558ms/step - accuracy: 0.7564 - loss: 2.5975 - precision_4: 0.7645 - recall_4: 0.7425
Epoch 11: val_accuracy improved from 0.78497 to 0.83716, saving model to /content/models/best_model_efficientnet_s1.h5


...

  class1_arc accuracy: 0.8187
  class2_whorl accuracy: 0.8742
  class3_loop accuracy: 0.7375
53/53 ━━━━━━━━━━━━━━━━━━━━ 36s 680ms/step - accuracy: 0.7343 - loss: 3.4842 - precision_6: 0.7457 - recall_6: 0.7171 - val_accuracy: 0.8100 - val_loss: 2.8216 - val_precision_6: 0.8265 - val_recall_6: 0.7954 - learning_rate: 0.0010
Epoch 10/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 597ms/step - accuracy: 0.7090 - loss: 3.0696 - precision_6: 0.7193 - recall_6: 0.6782
Epoch 10: val_accuracy improved from 0.81002 to 0.81211, saving model to /content/models/best_model_inceptionv3_s1.h5


  class1_arc accuracy: 0.9500
  class2_whorl accuracy: 0.7673
  class3_loop accuracy: 0.7188
53/53 ━━━━━━━━━━━━━━━━━━━━ 37s 699ms/step - accuracy: 0.7092 - loss: 3.0676 - precision_6: 0.7195 - recall_6: 0.6782 - val_accuracy: 0.8121 - val_loss: 2.4461 - val_precision_6: 0.8274 - val_recall_6: 0.7808 - learning_rate: 0.0010
Epoch 11/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 576ms/step - accuracy: 0.7292 - loss: 2.6432 - precision_6: 0.7436 - recall_6: 0.7016
Epoch 11: val_accuracy did not improve from 0.81211
  class1_arc accuracy: 0.8875
  class2_whorl accuracy: 0.5975
  class3_loop accuracy: 0.7500
53/53 ━━━━━━━━━━━━━━━━━━━━ 36s 685ms/step - accuracy: 0.7293 - loss: 2.6416 - precision_6: 0.7437 - recall_6: 0.7018 - val_accuracy: 0.7453 - val_loss: 2.2018 - val_precision_6: 0.7537 - val_recall_6: 0.7411 - learning_rate: 0.0010
Epoch 12/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 572ms/step - accuracy: 0.7029 - loss: 2.3759 - precision_6: 0.7158 - recall_6: 0.6790
Epoch 12: val_accuracy did not improve fro

  class1_arc accuracy: 0.7625
  class2_whorl accuracy: 0.8679
  class3_loop accuracy: 0.8500
53/53 ━━━━━━━━━━━━━━━━━━━━ 36s 688ms/step - accuracy: 0.7222 - loss: 1.7155 - precision_6: 0.7374 - recall_6: 0.6889 - val_accuracy: 0.8267 - val_loss: 1.3331 - val_precision_6: 0.8280 - val_recall_6: 0.8142 - learning_rate: 0.0010
Epoch 16/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 567ms/step - accuracy: 0.7405 - loss: 1.5546 - precision_6: 0.7527 - recall_6: 0.7187
Epoch 16: val_accuracy did not improve from 0.82672
  class1_arc accuracy: 0.9313
  class2_whorl accuracy: 0.6918
  class3_loop accuracy: 0.7375
53/53 ━━━━━━━━━━━━━━━━━━━━ 35s 664ms/step - accuracy: 0.7407 - loss: 1.5539 - precision_6: 0.7528 - recall_6: 0.7189 - val_accuracy: 0.7871 - val_loss: 1.2336 - val_precision_6: 0.7898 - val_recall_6: 0.7766 - learning_rate: 0.0010
Epoch 17/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 583ms/step - accuracy: 0.7362 - loss: 1.4404 - precision_6: 0.7555 - recall_6: 0.7233
Epoch 17: val_accuracy improved from 0.826

  class1_arc accuracy: 0.8812
  class2_whorl accuracy: 0.8616
  class3_loop accuracy: 0.7688
53/53 ━━━━━━━━━━━━━━━━━━━━ 37s 689ms/step - accuracy: 0.7363 - loss: 1.4397 - precision_6: 0.7555 - recall_6: 0.7234 - val_accuracy: 0.8372 - val_loss: 1.0904 - val_precision_6: 0.8450 - val_recall_6: 0.8309 - learning_rate: 0.0010
Epoch 18/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 592ms/step - accuracy: 0.7200 - loss: 1.4326 - precision_6: 0.7351 - recall_6: 0.6920
Epoch 18: val_accuracy did not improve from 0.83716
  class1_arc accuracy: 0.7875
  class2_whorl accuracy: 0.7610
  class3_loop accuracy: 0.8875
53/53 ━━━━━━━━━━━━━━━━━━━━ 36s 688ms/step - accuracy: 0.7200 - loss: 1.4320 - precision_6: 0.7352 - recall_6: 0.6921 - val_accuracy: 0.8121 - val_loss: 1.0547 - val_precision_6: 0.8195 - val_recall_6: 0.8058 - learning_rate: 0.0010
Epoch 19/30
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 577ms/step - accuracy: 0.7415 - loss: 1.2745 - precision_6: 0.7486 - recall_6: 0.7298
Epoch 19: val_accuracy did not improve fro

  class1_arc accuracy: 0.9000
  class2_whorl accuracy: 0.9119
  class3_loop accuracy: 0.6625
53/53 ━━━━━━━━━━━━━━━━━━━━ 124s 2s/step - accuracy: 0.5617 - loss: 1.7452 - precision_7: 0.5753 - recall_7: 0.5404 - val_accuracy: 0.8246 - val_loss: 0.8421 - val_precision_7: 0.8348 - val_recall_7: 0.7912 - learning_rate: 5.0000e-06
Epoch 2/25
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 595ms/step - accuracy: 0.6080 - loss: 1.4810 - precision_7: 0.6281 - recall_7: 0.5864
Epoch 2: val_accuracy did not improve from 0.82463
  class1_arc accuracy: 0.8688
  class2_whorl accuracy: 0.8868
  class3_loop accuracy: 0.6188
53/53 ━━━━━━━━━━━━━━━━━━━━ 36s 676ms/step - accuracy: 0.6082 - loss: 1.4812 - precision_7: 0.6283 - recall_7: 0.5865 - val_accuracy: 0.7912 - val_loss: 0.8634 - val_precision_7: 0.8061 - val_recall_7: 0.7724 - learning_rate: 5.0000e-06
Epoch 3/25
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 577ms/step - accuracy: 0.6322 - loss: 1.3966 - precision_7: 0.6403 - recall_7: 0.6032
Epoch 3: val_accuracy did not improve f

In [19]:
# ============================================
# SECTION 11: ADVANCED ENSEMBLE EVALUATION
# ============================================

print("\n" + "="*50)
print("ADVANCED ENSEMBLE MODEL EVALUATION")
print("="*50)

# Custom objects for loading (preserving original fix)
custom_objects = {
    'eff_preprocess_wrapper': eff_preprocess_wrapper,
    'inc_preprocess_wrapper': inc_preprocess_wrapper
}

# Load best models
print("Loading optimized models...")
best_eff_model = tf.keras.models.load_model(
    '/content/models/best_model_efficientnet_s2_finetuned.h5',
    custom_objects=custom_objects,
    compile=False
)
best_inc_model = tf.keras.models.load_model(
    '/content/models/best_model_inceptionv3_s2_finetuned.h5',
    custom_objects=custom_objects,
    compile=False
)
print("✓ Models loaded successfully!")

# Get predictions with Test-Time Augmentation
print("\nGenerating predictions with TTA...")
tta_eff = TestTimeAugmentation(best_eff_model, augmentations=3)
tta_inc = TestTimeAugmentation(best_inc_model, augmentations=3)

predictions_eff = []
predictions_inc = []

for i in range(len(test_generator)):
    batch = test_generator[i][0]
    for img in batch:
        predictions_eff.append(tta_eff.predict(img)[0])
        predictions_inc.append(tta_inc.predict(img)[0])
    if (i + 1) % 10 == 0:
        print(f"  Processed {(i+1)*BATCH_SIZE} images...")

predictions_eff = np.array(predictions_eff[:len(test_generator.classes)])
predictions_inc = np.array(predictions_inc[:len(test_generator.classes)])

# Weighted ensemble
ensemble_predictions = (ENSEMBLE_WEIGHTS['efficientnet'] * predictions_eff +
                       ENSEMBLE_WEIGHTS['inception'] * predictions_inc)

ensemble_predicted_classes = np.argmax(ensemble_predictions, axis=1)
true_classes = test_generator.classes
class_names = list(train_generator.class_indices.keys())


ADVANCED ENSEMBLE MODEL EVALUATION
Loading optimized models...
✓ Models loaded successfully!

Generating predictions with TTA...


In [20]:
# ============================================
# SECTION 12: COMPREHENSIVE ANALYSIS
# ============================================

def comprehensive_analysis(true_classes, predicted_classes, predictions, class_names):
    """Perform extensive analysis of results"""

    results = {}

    # Basic metrics
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

    results['accuracy'] = accuracy_score(true_classes, predicted_classes)
    results['precision'] = precision_score(true_classes, predicted_classes, average='weighted')
    results['recall'] = recall_score(true_classes, predicted_classes, average='weighted')
    results['f1'] = f1_score(true_classes, predicted_classes, average='weighted')

    print("\n" + "="*50)
    print("ENSEMBLE PERFORMANCE METRICS")
    print("="*50)
    print(f"Overall Accuracy: {results['accuracy']:.4f}")
    print(f"Weighted Precision: {results['precision']:.4f}")
    print(f"Weighted Recall: {results['recall']:.4f}")
    print(f"Weighted F1-Score: {results['f1']:.4f}")

    # Detailed classification report
    print("\n" + "="*50)
    print("DETAILED CLASSIFICATION REPORT")
    print("="*50)
    report = classification_report(true_classes, predicted_classes,
                                  target_names=class_names, digits=4, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    print(report_df)

    # Confusion Matrix Analysis
    cm = confusion_matrix(true_classes, predicted_classes)

    # Per-class analysis
    print("\n" + "="*50)
    print("PER-CLASS DETAILED ANALYSIS")
    print("="*50)

    for i, class_name in enumerate(class_names):
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        fp = np.sum(cm[:, i]) - tp
        tn = np.sum(cm) - tp - fn - fp

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

        print(f"\n{class_name}:")
        print(f"  True Positives: {tp}")
        print(f"  False Positives: {fp}")
        print(f"  False Negatives: {fn}")
        print(f"  Sensitivity (Recall): {sensitivity:.4f}")
        print(f"  Specificity: {specificity:.4f}")

        # Most confused with
        if fn > 0:
            confused_classes = []
            for j in range(len(class_names)):
                if i != j and cm[i, j] > 0:
                    confused_classes.append((class_names[j], cm[i, j]))
            if confused_classes:
                confused_classes.sort(key=lambda x: x[1], reverse=True)
                print(f"  Most confused with: {confused_classes[0][0]} ({confused_classes[0][1]} samples)")

    # Confidence Analysis
    print("\n" + "="*50)
    print("CONFIDENCE ANALYSIS")
    print("="*50)

    max_confidences = np.max(predictions, axis=1)
    correct_mask = predicted_classes == true_classes

    correct_conf = max_confidences[correct_mask]
    incorrect_conf = max_confidences[~correct_mask]

    print(f"Mean confidence (correct): {np.mean(correct_conf):.4f} ± {np.std(correct_conf):.4f}")
    print(f"Mean confidence (incorrect): {np.mean(incorrect_conf):.4f} ± {np.std(incorrect_conf):.4f}")

    # Statistical test for confidence difference
    t_stat, p_value = stats.ttest_ind(correct_conf, incorrect_conf)
    print(f"T-test for confidence difference: t={t_stat:.4f}, p={p_value:.6f}")

    # Threshold analysis
    thresholds = np.arange(0.5, 1.0, 0.05)
    threshold_results = []

    for threshold in thresholds:
        high_conf_mask = max_confidences >= threshold
        if np.sum(high_conf_mask) > 0:
            high_conf_acc = np.mean(predicted_classes[high_conf_mask] == true_classes[high_conf_mask])
            coverage = np.mean(high_conf_mask)
            threshold_results.append({
                'threshold': threshold,
                'accuracy': high_conf_acc,
                'coverage': coverage
            })

    threshold_df = pd.DataFrame(threshold_results)
    print("\nConfidence Threshold Analysis:")
    print(threshold_df)

    return results, cm, report_df, threshold_df

# Perform comprehensive analysis
results, cm, report_df, threshold_df = comprehensive_analysis(
    true_classes, ensemble_predicted_classes, ensemble_predictions, class_names
)


ENSEMBLE PERFORMANCE METRICS
Overall Accuracy: 0.8971
Weighted Precision: 0.8982
Weighted Recall: 0.8971
Weighted F1-Score: 0.8965

DETAILED CLASSIFICATION REPORT
              precision    recall  f1-score     support
class1_arc     0.949367  0.925926  0.937500   81.000000
class2_whorl   0.865169  0.950617  0.905882   81.000000
class3_loop    0.880000  0.814815  0.846154   81.000000
accuracy       0.897119  0.897119  0.897119    0.897119
macro avg      0.898179  0.897119  0.896512  243.000000
weighted avg   0.898179  0.897119  0.896512  243.000000

PER-CLASS DETAILED ANALYSIS

class1_arc:
  True Positives: 75
  False Positives: 4
  False Negatives: 6
  Sensitivity (Recall): 0.9259
  Specificity: 0.9753
  Most confused with: class3_loop (5 samples)

class2_whorl:
  True Positives: 77
  False Positives: 12
  False Negatives: 4
  Sensitivity (Recall): 0.9506
  Specificity: 0.9259
  Most confused with: class3_loop (4 samples)

class3_loop:
  True Positives: 66
  False Positives: 9
  Fals